# Diagnostic du checkpoint CSP (sans conduite)

Executer les deux cellules ci-dessous, jeu ouvert dans une session locale. 
Elles interrogent uniquement PING, CAPABILITIES et STATUS. Aucun besoin de vJoy, PPO, 
camera ou telemetrie. Un retour PONG ne prouve pas la disponibilite des checkpoints.

Si les deux capacites sont false, verifier CSP et relancer ce diagnostic apres 
tout changement. Si STATUS vaut NOT_SAVED, enregistrer le point dans le notebook 03.02 
apres avoir obtenu une capacite utilisable.


In [3]:
# Cellule de pont entre Jupyter et l'application CSP Lua OUARTICheckpoint.
import json, os, time, uuid, math
from pathlib import Path

BRIDGE_DIR = Path.home() / 'Documents' / 'Assetto Corsa' / 'OUARTI_checkpoint_v3'
ANCHOR_FILE = BRIDGE_DIR / 'anchor.json'

class CheckpointBridge:
    def __init__(self, directory=BRIDGE_DIR):
        self.directory = Path(directory)
        self.command = self.directory / 'command.txt'
        self.response = self.directory / 'status.txt'

    def request(self, command, timeout=8.0):
        if command not in ('PING', 'STATUS', 'SAVE', 'LOAD', 'CAPABILITIES'):
            raise ValueError('Commande CSP inconnue: ' + command)
        if not self.directory.is_dir():
            raise RuntimeError('Application CSP OUARTICheckpoint absente : ' + str(self.directory))
        token = uuid.uuid4().hex
        temporary = self.directory / ('command_' + token + '.tmp')
        temporary.write_text(token + '|' + command, encoding='ascii')
        os.replace(str(temporary), str(self.command))
        deadline = time.monotonic() + float(timeout)
        last = ''
        while time.monotonic() < deadline:
            try:
                last = self.response.read_text(encoding='utf8').strip()
            except (FileNotFoundError, UnicodeError, PermissionError):
                time.sleep(0.05)
                continue
            parts = last.split('|', 2)
            if len(parts) == 3 and parts[0] == token:
                if parts[1] == 'ERROR':
                    raise RuntimeError('CSP checkpoint: ' + parts[2])
                if parts[1] == 'PENDING':
                    time.sleep(0.05)
                    continue
                return parts[1], parts[2]
            time.sleep(0.05)
        raise TimeoutError('Pas de reponse CSP pour ' + command + '. Dernier statut: ' + last[:140])

    def capabilities(self):
        code, detail = self.request('CAPABILITIES')
        if code != 'CAPABILITIES':
            raise RuntimeError('Reponse CAPABILITIES inattendue: ' + code + ' ' + detail)
        fields = {}
        for field in detail.split(';'):
            key, separator, value = field.strip().partition('=')
            if separator:
                fields[key.strip()] = value.strip()
        for key in ('offline', 'full_snapshot', 'position_fallback'):
            if fields.get(key) not in ('true', 'false'):
                raise RuntimeError('Capacite CSP absente ou invalide: ' + key + ': ' + detail)
        return {key: value == 'true' for key, value in fields.items()
                if key in ('offline', 'full_snapshot', 'position_fallback')}

    def assert_supported(self):
        caps = self.capabilities()
        if not caps['offline']:
            raise RuntimeError('Checkpoint reserve a une session locale hors ligne.')
        if not (caps['full_snapshot'] or caps['position_fallback']):
            raise RuntimeError(
                'Checkpoint indisponible: CSP ne fournit ni snapshot complet ni '
                'repositionnement autorise dans cette session. La connexion au pont fonctionne. '
                'Verifier la version CSP dans Content Manager > Settings > Custom Shaders Patch '
                '> About & Updates. Sur le PC examine le 16/11/2026, CSP 0.1.79 est installe '
                'et son SDK ne declare pas saveCarStateAsync/loadCarState. '
                'Apres mise a jour compatible avec sensors_par, relancer le jeu et refaire '
                'PING puis CAPABILITIES. Si les deux capacites restent false, ne pas lancer PPO. '
                'Le fallback position depend aussi de physics.allowed(); '
                'une mise a jour seule ne garantit pas son activation.')
        return caps

    def assert_saved(self):
        code, detail = self.request('STATUS')
        if code != 'SAVED':
            raise RuntimeError('Checkpoint non enregistre dans cette session CSP: ' + detail
                               + '. Executer la cellule de sauvegarde avec ACTIVER_SAUVEGARDE_CHECKPOINT=True, '
                               'voiture immobile, puis remettre False. Refaire SAVE apres chaque redemarrage du jeu.')
        return detail

    def load(self):
        code, detail = self.request('LOAD', timeout=8)
        if code != 'LOAD_REQUESTED':
            raise RuntimeError('CSP n a pas confirme la commande LOAD: ' + code + ' ' + detail)


def telemetry_xy(raw):
    return float(raw['world_position_x']), float(raw['world_position_y'])


def distance_from_anchor(raw, anchor):
    x, y = telemetry_xy(raw)
    xa, ya = float(anchor['x']), float(anchor['y'])
    distance = math.hypot(x - xa, y - ya)
    angle_error = abs((float(raw['yaw']) - float(anchor['yaw']) + math.pi) % (2 * math.pi) - math.pi)
    return distance, angle_error


def make_anchor_from_raw(raw, track, car):
    x, y = telemetry_xy(raw)
    if float(raw.get('speed', 999)) > 0.3:
        raise RuntimeError('Immobiliser la voiture avant sauvegarde (<1 km/h).')
    return {'track': track, 'car': car, 'x': x, 'y': y,
            'yaw': float(raw['yaw']), 'created_at': time.time(),
            'note': 'Checkpoint CSP en memoire, non reutilisable apres redemarrage du jeu.'}


In [4]:
# Diagnostic en lecture seule : aucun SAVE/LOAD et aucune commande de conduite.
bridge = CheckpointBridge()
print('Application :', bridge.request('PING'))
print('Fonctions CSP disponibles :', bridge.capabilities())
try:
    print('Compatibilite checkpoint :', bridge.assert_supported())
    print('Checkpoint :', bridge.request('STATUS'))
except RuntimeError as exc:
    print('ENTRAINEMENT INDISPONIBLE :', exc)


Application : ('PONG', 'OUARTI CSP app active - V3.2')
Fonctions CSP disponibles : {'full_snapshot': True, 'position_fallback': False, 'offline': True}
Compatibilite checkpoint : {'full_snapshot': True, 'position_fallback': False, 'offline': True}
Checkpoint : ('NOT_SAVED', 'PAS DE CHECKPOINT')
